# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:22<00:00,  7.62s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Open-box TCL NXTPAPER 11 Plus 11" 256GB Android Tablet for $158 + free shipping\nDetails: Use promo code "VIPOUTLETMAY26" to get the open-box TCL NXTPAPER 11 Plus 11" 256GB Android Tablet for $158.40. That\'s $18 less than our mention from a month ago and the best deal we\'ve seen for this tablet. Coupon expires June 1. Buy Now at eBay\nFeatures: \nURL: https://www.dealnews.com/Open-box-TCL-NXTPAPER-11-Plus-11-256-GB-Android-Tablet-for-158-free-shipping/21825628.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Easy One Touch Mini CD Player Phone Mount for $14 + free shipping w/ $35
Details: Walmart offers the Easy One Touch Mini CD Player Phone Mount for $13.57.   That's a $6 low.  Choose pickup or spend $35 to avoid the $6.99 shipping charge. Buy Now at Walmart
Features: 
URL: https://www.dealnews.com/Easy-One-Touch-Mini-CD-Player-Phone-Mount-for-14-free-shipping-w-35/21825640.html?

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Marshall Major IV are on-ear Bluetooth headphones offering long battery life and portable comfort. The model delivers up to 80+ hours of listening time on a single full charge, wireless Bluetooth connectivity, and Marshall’s signature tuned sound in a compact, foldable design. The headphones are finished with classic Marshall styling and controls for playback and calls on the earcup.', price=59.99, url='https://www.dealnews.com/products/Marshall-Amplification/Marshall-Major-IV-On-Ear-Bluetooth-Headphones/278321.html?iref=rss-c142'), Deal(product_description='Segway Lumina 500 is a 512 Wh portable power station designed for off-grid power needs and emergency backup. It provides multiple AC, DC, and USB outputs to run small appliances, charge laptops, phones, and power camping equipment. The unit balances capacity and portability for home backup, travel, and outdoor use, with integrated safety and charging options.', price=153.0, url='https:

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Marshall Major IV are on-ear Bluetooth headphones offering long battery life and portable comfort. The model delivers up to 80+ hours of listening time on a single full charge, wireless Bluetooth connectivity, and Marshall’s signature tuned sound in a compact, foldable design. The headphones are finished with classic Marshall styling and controls for playback and calls on the earcup.
59.99
https://www.dealnews.com/products/Marshall-Amplification/Marshall-Major-IV-On-Ear-Bluetooth-Headphones/278321.html?iref=rss-c142

Segway Lumina 500 is a 512 Wh portable power station designed for off-grid power needs and emergency backup. It provides multiple AC, DC, and USB outputs to run small appliances, charge laptops, phones, and power camping equipment. The unit balances capacity and portability for home backup, travel, and outdoor use, with integrated safety and charging options.
153.0
https://www.dealnews.com/Segway-Lumina-500-512-Wh-Portable-Power-Station-for-153-free-shipping/21825618.html?

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='Marshall Major IV is an on-ear Bluetooth headphone featuring up to 80+ hours of listening time per charge, classic Marshall aesthetic with tactile controls, and a compact foldable design for portability. It connects wirelessly for music and calls and includes rechargeable battery and likely auxiliary wired use. The model listed is 1005773 and targets users wanting long battery life in a vintage-style, comfortable on-ear headset.', price=59.99, url='https://www.dealnews.com/products/Marshall-Amplification/Marshall-Major-IV-On-Ear-Bluetooth-Headphones/278321.html?iref=rss-c142'), Deal(product_description='Segway Lumina 500 is a 512Wh portable power station designed to provide off-grid power for camping, emergencies, and outdoor use. It includes multiple output ports for charging phones, laptops, and small appliances, onboard battery management for safety, and a mid-sized capacity suited for running essential devices for extended periods with

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [21]:
load_dotenv(override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [19]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [24]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
19:55:50 - LiteLLM:INFO: utils.py:3748 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
19:55:53 - LiteLLM:INFO: utils.py:1573 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
